In [ ]:
#-------------------------------------TensorFlow-------------------------------------#

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
import os
import matplotlib.pyplot as plt

def MSCFF_V2(imagesize, imageDir, labelDir, imageDirV, labelDirV):
    """
    A full convolution neural network model based on multi-scale feature fusion for cloud area detection
    of remote sensing images implemented in TensorFlow/Keras.
    
    Parameters:
    - imagesize: Tuple of (height, width, channels) for input images
    - imageDir: Path to training image directory
    - labelDir: Path to training label directory
    - imageDirV: Path to validation image directory
    - labelDirV: Path to validation label directory
    
    Returns:
    - model: Compiled Keras model
    - history: Training history
    """
    
    # Input layer
    inputs = layers.Input(shape=imagesize, name="imageinput")
    
    # ------------------- Encoder Path -------------------
    # CBRR Block 1
    conv_1 = layers.Conv2D(64, (3, 3), padding='same', name="conv_1")(inputs)
    batchnorm_1 = layers.BatchNormalization(name="batchnorm_1")(conv_1)
    relu_1 = layers.ReLU(name="relu_1")(batchnorm_1)
    
    conv_2 = layers.Conv2D(64, (3, 3), padding='same', name="conv_2")(relu_1)
    batchnorm_2 = layers.BatchNormalization(name="batchnorm_2")(conv_2)
    relu_2 = layers.ReLU(name="relu_2")(batchnorm_2)
    
    conv_3 = layers.Conv2D(64, (3, 3), padding='same', name="conv_3")(relu_2)
    batchnorm_3 = layers.BatchNormalization(name="batchnorm_3")(conv_3)
    relu_3 = layers.ReLU(name="relu_3")(batchnorm_3)
    
    addition_1 = layers.Add(name="addition_1")([relu_3, relu_1])
    
    # Maxpool layer 1
    maxpool_1 = layers.MaxPooling2D((2, 2), strides=(2, 2), name="maxpool_1")(addition_1)

    # CBRR Block 2
    conv_4 = layers.Conv2D(128, (3, 3), padding='same', name="conv_4")(maxpool_1)
    batchnorm_4 = layers.BatchNormalization(name="batchnorm_4")(conv_4)
    relu_4 = layers.ReLU(name="relu_4")(batchnorm_4)
    
    conv_5 = layers.Conv2D(128, (3, 3), padding='same', name="conv_5")(relu_4)
    batchnorm_5 = layers.BatchNormalization(name="batchnorm_5")(conv_5)
    relu_5 = layers.ReLU(name="relu_5")(batchnorm_5)
    
    conv_6 = layers.Conv2D(128, (3, 3), padding='same', name="conv_6")(relu_5)
    batchnorm_6 = layers.BatchNormalization(name="batchnorm_6")(conv_6)
    relu_6 = layers.ReLU(name="relu_6")(batchnorm_6)
    
    addition_2 = layers.Add(name="addition_2")([relu_6, relu_4])
    
    # Maxpool layer 2
    maxpool_2 = layers.MaxPooling2D((2, 2), strides=(2, 2), name="maxpool_2")(addition_2)
    
    # CBRR Block 3
    conv_7 = layers.Conv2D(256, (3, 3), padding='same', name="conv_7")(maxpool_2)
    batchnorm_7 = layers.BatchNormalization(name="batchnorm_7")(conv_7)
    relu_7 = layers.ReLU(name="relu_7")(batchnorm_7)
    
    conv_8 = layers.Conv2D(256, (3, 3), padding='same', name="conv_8")(relu_7)
    batchnorm_8 = layers.BatchNormalization(name="batchnorm_8")(conv_8)
    relu_8 = layers.ReLU(name="relu_8")(batchnorm_8)
    
    conv_9 = layers.Conv2D(256, (3, 3), padding='same', name="conv_9")(relu_8)
    batchnorm_9 = layers.BatchNormalization(name="batchnorm_9")(conv_9)
    relu_9 = layers.ReLU(name="relu_9")(batchnorm_9)
    
    addition_3 = layers.Add(name="addition_3")([relu_9, relu_7])
    
    # Maxpool layer 3
    xmaxpool_3 = layers.MaxPooling2D((2, 2), strides=(2, 2), name="maxpool_3")(addition_3)
    
    # CBRR Block 4
    conv_10 = layers.Conv2D(512, (3, 3), padding='same', name="conv_10")(maxpool_3)
    batchnorm_10 = layers.BatchNormalization(name="batchnorm_10")(conv_10)
    relu_10 = layers.ReLU(name="relu_10")(batchnorm_10)
    
    conv_11 = layers.Conv2D(512, (3, 3), padding='same', name="conv_11")(relu_10)
    batchnorm_11 = layers.BatchNormalization(name="batchnorm_11")(conv_11)
    relu_11 = layers.ReLU(name="relu_11")(batchnorm_11)
    
    conv_12 = layers.Conv2D(512, (3, 3), padding='same', name="conv_12")(relu_11)
    batchnorm_12 = layers.BatchNormalization(name="batchnorm_12")(conv_12)
    relu_12 = layers.ReLU(name="relu_12")(batchnorm_12)
    
    addition_4 = layers.Add(name="addition_4")([relu_12, relu_10])
    


    # ------------------- Multi-scale Feature Fusion Path -------------------
    # Path 1
    conv_13 = layers.Conv2D(512, (3, 3), padding='same', name="conv_13")(addition_4)
    batchnorm_13 = layers.BatchNormalization(name="batchnorm_13")(conv_13)
    relu_13 = layers.ReLU(name="relu_13")(batchnorm_13)
    
    conv_14 = layers.Conv2D(512, (3, 3), padding='same', dilation_rate=2, name="conv_14")(relu_13)
    batchnorm_14 = layers.BatchNormalization(name="batchnorm_14")(conv_14)
    relu_14 = layers.ReLU(name="relu_14")(batchnorm_14)
    
    conv_15 = layers.Conv2D(512, (3, 3), padding='same', dilation_rate=5, name="conv_15")(relu_14)
    batchnorm_15 = layers.BatchNormalization(name="batchnorm_15")(conv_15)
    relu_15 = layers.ReLU(name="relu_15")(batchnorm_15)
    
    addition_5 = layers.Add(name="addition_5")([relu_15, relu_13])
    
    conv_16 = layers.Conv2D(512, (3, 3), padding='same', name="conv_16")(addition_5)
    batchnorm_16 = layers.BatchNormalization(name="batchnorm_16")(conv_16)
    relu_16 = layers.ReLU(name="relu_16")(batchnorm_16)
    
    conv_17 = layers.Conv2D(512, (3, 3), padding='same', dilation_rate=3, name="conv_17")(relu_16)
    batchnorm_17 = layers.BatchNormalization(name="batchnorm_17")(conv_17)
    relu_17 = layers.ReLU(name="relu_17")(batchnorm_17)
    
    conv_18 = layers.Conv2D(512, (3, 3), padding='same', dilation_rate=5, name="conv_18")(relu_17)
    batchnorm_18 = layers.BatchNormalization(name="batchnorm_18")(conv_18)
    relu_18 = layers.ReLU(name="relu_18")(batchnorm_18)
    
    addition_6 = layers.Add(name="addition_6")([relu_18, relu_16])
    
    conv_19 = layers.Conv2D(512, (3, 3), padding='same', name="conv_19")(addition_6)
    batchnorm_19 = layers.BatchNormalization(name="batchnorm_19")(conv_19)
    relu_19 = layers.ReLU(name="relu_19")(batchnorm_19)
    
    conv_20 = layers.Conv2D(512, (3, 3), padding='same', dilation_rate=3, name="conv_20")(relu_19)
    batchnorm_20 = layers.BatchNormalization(name="batchnorm_20")(conv_20)
    relu_20 = layers.ReLU(name="relu_20")(batchnorm_20)
    
    conv_21 = layers.Conv2D(512, (3, 3), padding='same', dilation_rate=5, name="conv_21")(relu_20)
    batchnorm_21 = layers.BatchNormalization(name="batchnorm_21")(conv_21)
    relu_21 = layers.ReLU(name="relu_21")(batchnorm_21)
    
    addition_7 = layers.Add(name="addition_7")([relu_21, relu_19])
    
    addition_9 = layers.Add(name="addition_9")([addition_7, addition_4])
    
    # Path 2
    conv_22 = layers.Conv2D(512, (3, 3), padding='same', name="conv_22")(addition_9)
    batchnorm_22 = layers.BatchNormalization(name="batchnorm_22")(conv_22)
    relu_22 = layers.ReLU(name="relu_22")(batchnorm_22)
    
    conv_23 = layers.Conv2D(512, (3, 3), padding='same', dilation_rate=2, name="conv_23")(relu_22)
    batchnorm_23 = layers.BatchNormalization(name="batchnorm_23")(conv_23)
    relu_23 = layers.ReLU(name="relu_23")(batchnorm_23)
    
    conv_24 = layers.Conv2D(512, (3, 3), padding='same', dilation_rate=5, name="conv_24")(relu_23)
    batchnorm_24 = layers.BatchNormalization(name="batchnorm_24")(conv_24)
    relu_24 = layers.ReLU(name="relu_24")(batchnorm_24)
    
    addition_8 = layers.Add(name="addition_8")([relu_24, relu_22])
    
    addition_10 = layers.Add(name="addition_10")([addition_8, addition_9])
    
    conv_25 = layers.Conv2D(512, (3, 3), padding='same', name="conv_25")(addition_10)
    batchnorm_25 = layers.BatchNormalization(name="batchnorm_25")(conv_25)
    relu_25 = layers.ReLU(name="relu_25")(batchnorm_25)
    
    conv_26 = layers.Conv2D(512, (3, 3), padding='same', name="conv_26")(relu_25)
    batchnorm_26 = layers.BatchNormalization(name="batchnorm_26")(conv_26)
    relu_26 = layers.ReLU(name="relu_26")(batchnorm_26)
    
    conv_27 = layers.Conv2D(512, (3, 3), padding='same', name="conv_27")(relu_26)
    batchnorm_27 = layers.BatchNormalization(name="batchnorm_27")(conv_27)
    relu_27 = layers.ReLU(name="relu_27")(batchnorm_27)
    
    addition_11 = layers.Add(name="addition_11")([relu_27, relu_25])
    
    addition_12 = layers.Add(name="addition_12")([addition_11, addition_10])
    
    
    # ------------------- Decoder Path -------------------
    # Block 1
    transposed_conv_1 = layers.Conv2DTranspose(512, (4, 4), strides=(2, 2), padding='same', 
                                             name="transposed-conv_1")(addition_12)
    conv_28 = layers.Conv2D(256, (3, 3), padding='same', name="conv_28")(transposed_conv_1)
    batchnorm_28 = layers.BatchNormalization(name="batchnorm_28")(conv_28)
    relu_28 = layers.ReLU(name="relu_28")(batchnorm_28)
    
    conv_29 = layers.Conv2D(256, (3, 3), padding='same', name="conv_29")(relu_28)
    batchnorm_29 = layers.BatchNormalization(name="batchnorm_29")(conv_29)
    relu_29 = layers.ReLU(name="relu_29")(batchnorm_29)
    
    conv_30 = layers.Conv2D(256, (3, 3), padding='same', name="conv_30")(relu_29)
    batchnorm_30 = layers.BatchNormalization(name="batchnorm_30")(conv_30)
    relu_30 = layers.ReLU(name="relu_30")(batchnorm_30)
    
    addition_13 = layers.Add(name="addition_13")([relu_30, relu_28])
    
    addition_14 = layers.Add(name="addition_14")([addition_13, addition_3])
    
    # Block 2
    transposed_conv_2 = layers.Conv2DTranspose(512, (4, 4), strides=(2, 2), padding='same', 
                                             name="transposed-conv_2")(addition_14)
    conv_31 = layers.Conv2D(128, (3, 3), padding='same', name="conv_31")(transposed_conv_2)
    batchnorm_31 = layers.BatchNormalization(name="batchnorm_31")(conv_31)
    relu_31 = layers.ReLU(name="relu_31")(batchnorm_31)
    
    conv_32 = layers.Conv2D(128, (3, 3), padding='same', name="conv_32")(relu_31)
    batchnorm_32 = layers.BatchNormalization(name="batchnorm_32")(conv_32)
    relu_32 = layers.ReLU(name="relu_32")(batchnorm_32)
    
    conv_33 = layers.Conv2D(128, (3, 3), padding='same', name="conv_33")(relu_32)
    batchnorm_33 = layers.BatchNormalization(name="batchnorm_33")(conv_33)
    relu_33 = layers.ReLU(name="relu_33")(batchnorm_33)
    
    addition_15 = layers.Add(name="addition_15")([relu_33, relu_31])
    
    addition_16 = layers.Add(name="addition_16")([addition_15, addition_2])
    
    # Block 3
    transposed_conv_3 = layers.Conv2DTranspose(512, (4, 4), strides=(2, 2), padding='same', 
                                             name="transposed-conv_3")(addition_16)
    conv_34 = layers.Conv2D(64, (3, 3), padding='same', name="conv_34")(transposed_conv_3)
    batchnorm_34 = layers.BatchNormalization(name="batchnorm_34")(conv_34)
    relu_34 = layers.ReLU(name="relu_34")(batchnorm_34)
    
    conv_35 = layers.Conv2D(64, (3, 3), padding='same', name="conv_35")(relu_34)
    batchnorm_35 = layers.BatchNormalization(name="batchnorm_35")(conv_35)
    relu_35 = layers.ReLU(name="relu_35")(batchnorm_35)
    
    conv_36 = layers.Conv2D(64, (3, 3), padding='same', name="conv_36")(relu_35)
    batchnorm_36 = layers.BatchNormalization(name="batchnorm_36")(conv_36)
    relu_36 = layers.ReLU(name="relu_36")(batchnorm_36)
    
    addition_18 = layers.Add(name="addition_18")([relu_36, relu_34])
    
    # ------------------- Output Path -------------------
    # Path 1
    conv_37 = layers.Conv2D(2, (3, 3), padding='same', name="conv_37")(addition_1)
    batchnorm_37 = layers.BatchNormalization(name="batchnorm_37")(conv_37)
    relu_37 = layers.ReLU(name="relu_37")(batchnorm_37)
    
    # Path 2
    transposed_conv_4 = layers.Conv2DTranspose(2, (4, 4), strides=(2, 2), padding='same', 
                                             name="transposed-conv_4")(addition_18)
    
    # Path 3
    conv_38 = layers.Conv2D(2, (3, 3), padding='same', name="conv_38")(addition_16)
    batchnorm_38 = layers.BatchNormalization(name="batchnorm_38")(conv_38)
    relu_38 = layers.ReLU(name="relu_38")(batchnorm_38)
    transposed_conv_5 = layers.Conv2DTranspose(2, (8, 8), strides=(4, 4), padding='same', 
                                             name="transposed-conv_5")(relu_38)
    
    # Path 4
    conv_39 = layers.Conv2D(2, (3, 3), padding='same', name="conv_39")(addition_14)
    batchnorm_39 = layers.BatchNormalization(name="batchnorm_39")(conv_39)
    relu_39 = layers.ReLU(name="relu_39")(batchnorm_39)
    transposed_conv_6 = layers.Conv2DTranspose(2, (16, 16), strides=(8, 8), padding='same', 
                                             name="transposed-conv_6")(relu_39)
    
    # Path 5
    conv_40 = layers.Conv2D(2, (3, 3), padding='same', name="conv_40")(addition_12)
    batchnorm_40 = layers.BatchNormalization(name="batchnorm_40")(conv_40)
    relu_40 = layers.ReLU(name="relu_40")(batchnorm_40)
    transposed_conv_7 = layers.Conv2DTranspose(2, (16, 16), strides=(8, 8), padding='same', 
                                             name="transposed-conv_7")(relu_40)
    
    # Path 6
    conv_41 = layers.Conv2D(2, (3, 3), padding='same', name="conv_41")(addition_10)
    batchnorm_41 = layers.BatchNormalization(name="batchnorm_41")(conv_41)
    relu_41 = layers.ReLU(name="relu_41")(batchnorm_41)
    transposed_conv_8 = layers.Conv2DTranspose(2, (16, 16), strides=(8, 8), padding='same', 
                                             name="transposed-conv_8")(relu_41)
    
    # Path 7
    conv_42 = layers.Conv2D(2, (3, 3), padding='same', name="conv_42")(addition_9)
    batchnorm_42 = layers.BatchNormalization(name="batchnorm_42")(conv_42)
    relu_42 = layers.ReLU(name="relu_42")(batchnorm_42)
    transposed_conv_9 = layers.Conv2DTranspose(2, (16, 16), strides=(8, 8), padding='same', 
                                             name="transposed-conv_9")(relu_42)
    
    # Concatenate all paths
    depthcat = layers.Concatenate(axis=-1, name="depthcat")(
        [relu_37, transposed_conv_4, transposed_conv_5, 
         transposed_conv_6, transposed_conv_7, transposed_conv_8]
    )
    
    # Final layers
    conv_43 = layers.Conv2D(2, (3, 3), padding='same', name="conv_43")(depthcat)
    softmax = layers.Softmax(name="softmax")(conv_43)
    
    # Create model
    model = Model(inputs=inputs, outputs=softmax, name="MSCFF_V2")
    
    # ------------------- Data Preparation -------------------
    # Note: In Python, we typically use different data loading approaches
    # Here's a simplified version. You might need to adapt based on your actual data format
    
    # Define class names and label mappings
    class_names = ["background", "cloud"]
    num_classes = len(class_names)
    
    # Create training dataset
    def load_and_preprocess_image(image_path, label_path):
        image = tf.io.read_file(image_path)
        image = tf.image.decode_png(image, channels=3)
        image = tf.image.convert_image_dtype(image, tf.float32)
        
        label = tf.io.read_file(label_path)
        label = tf.image.decode_png(label, channels=1)
        # Convert labels to binary (0 or 1)
        label = tf.where(label > 0, 1, 0)
        label = tf.one_hot(label[..., 0], depth=num_classes)
        
        return image, label
    
    # Get list of image and label files
    train_image_files = sorted([os.path.join(imageDir, f) for f in os.listdir(imageDir)])
    train_label_files = sorted([os.path.join(labelDir, f) for f in os.listdir(labelDir)])
    
    val_image_files = sorted([os.path.join(imageDirV, f) for f in os.listdir(imageDirV)])
    val_label_files = sorted([os.path.join(labelDirV, f) for f in os.listdir(labelDirV)])
    
    # Create TensorFlow datasets
    train_dataset = tf.data.Dataset.from_tensor_slices((train_image_files, train_label_files))
    train_dataset = train_dataset.map(load_and_preprocess_image)
    train_dataset = train_dataset.batch(1).prefetch(tf.data.AUTOTUNE)
    
    val_dataset = tf.data.Dataset.from_tensor_slices((val_image_files, val_label_files))
    val_dataset = val_dataset.map(load_and_preprocess_image)
    val_dataset = val_dataset.batch(1).prefetch(tf.data.AUTOTUNE)
    
    # ------------------- Training Setup -------------------
    # Create checkpoint directory
    checkpoint_path = os.path.join(os.getcwd(), 'checkpoint')
    if not os.path.exists(checkpoint_path):
        os.makedirs(checkpoint_path)
    
    # Callbacks
    checkpoint_callback = ModelCheckpoint(
        filepath=os.path.join(checkpoint_path, 'model-{epoch:02d}.h5'),
        save_weights_only=False,
        save_best_only=True,
        monitor='val_loss',
        mode='min'
    )
    
    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    )
    
    # Compile model
    model.compile(
        optimizer=SGD(learning_rate=0.01, momentum=0.9),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    # Train model
    history = model.fit(
        train_dataset,
        epochs=30,
        validation_data=val_dataset,
        callbacks=[checkpoint_callback, reduce_lr]
    )
    
    return model, history

# Example usage:
# model, history = MSCFF_V2((512, 512, 3), 'train/images', 'train/labels', 'val/images', 'val/labels')